# 🚀 Modern Data Stack no Google Colab: dbt Core + DuckDB
## Pipeline Modular de Transformação e Qualidade de Dados (Dataset Olist E-Commerce)

---

### 📌 1. Apresentação e Objetivo do Case
Neste laboratório prático, implementaremos uma arquitetura analítica moderna e *serverless* utilizando **dbt Core** integrado nativamente ao **DuckDB** (`dbt-duckdb`).

O objetivo é transformar os dados brutos de e-commerce da **Olist** (maior marketplace do Brasil, com mais de 100 mil pedidos reais entre 2016 e 2018) em uma camada analítica consolidada e confiável, aplicando:
- **Modularidade e Governança:** Divisão estruturada nas camadas *Staging*, *Intermediate* e *Marts*.
- **Data Quality & Testing:** Garantia automatizada de unicidade, não-nulidade e integridade de domínio de valores.
- **Alta Performance In-Process:** Processamento analítico colunar veloz com DuckDB, sem dependência de data warehouses em nuvem de alto custo.

---

### 🏛️ 2. Arquitetura da Solução

```
[ Kaggle Dataset (Olist) ]
            │
            ▼ (kagglehub)
   [ raw_data/*.csv ]
            │
    ┌───────┴─────────────────────────────────────────┐
    │                dbt-duckdb                       │
    │                                                 │
    │  1. Camada Staging (stg_)                       │
    │     - Leitura das fontes raw                    │
    │     - Limpeza, tipagem (casting) e renomeação   │
    │                                                 │
    │  2. Camada Intermediate (int_)                  │
    │     - Agregação por pedido (itens, frete, etc.) │
    │                                                 │
    │  3. Camada Marts (fct_)                         │
    │     - Tabela Fato de Pedidos (fct_orders)       │
    │     - Métricas consolidadas e lead time         │
    │                                                 │
    │  4. Data Quality Tests                          │
    │     - unique, not_null, accepted_values         │
    └───────────────────────┬─────────────────────────┘
                            │
                            ▼
           [ olist_analytics.duckdb (Arquivo OLAP) ]
                            │
                            ▼
              [ Análise / BI em Python ]
```

In [ ]:
# ==============================================================================
# Célula 2: Instalação das Dependências no Ambiente Colab
# ==============================================================================
# Instalamos dbt-duckdb (que provê dbt-core e duckdb), pandas e kagglehub
!pip install -q dbt-duckdb duckdb pandas kagglehub tabulate

In [ ]:
# ==============================================================================
# Célula 3: Ingestão Automatizada e Organização dos Dados Brutos (Olist)
# ==============================================================================
import os
import shutil
import kagglehub

print("📥 Baixando dataset público Olist Brazilian E-Commerce via kagglehub...")
dataset_download_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(f"✅ Download concluído no cache: {dataset_download_path}")

# Definir e criar o diretório raw_data local
base_dir = "/content" if os.path.exists("/content") else os.path.abspath(".")
raw_data_dir = os.path.join(base_dir, "raw_data")
os.makedirs(raw_data_dir, exist_ok=True)

# Copiar os arquivos CSV relevantes para raw_data/
copied_files = []
for file_name in os.listdir(dataset_download_path):
    if file_name.endswith(".csv"):
        src = os.path.join(dataset_download_path, file_name)
        dst = os.path.join(raw_data_dir, file_name)
        shutil.copy(src, dst)
        copied_files.append(file_name)

print(f"\n📂 {len(copied_files)} arquivos CSV organizados com sucesso em '{raw_data_dir}':")
for file_name in sorted(copied_files):
    size_mb = os.path.getsize(os.path.join(raw_data_dir, file_name)) / (1024 * 1024)
    print(f"  • {file_name} ({size_mb:.2f} MB)")

In [ ]:
# ==============================================================================
# Célula 4: Estruturação dos Diretórios dbt e Configuração de Perfis
# ==============================================================================
import os

base_dir = "/content" if os.path.exists("/content") else os.path.abspath(".")
dbt_project_dir = os.path.join(base_dir, "olist_dbt")
db_path = os.path.join(base_dir, "olist_analytics.duckdb").replace("\\", "/")

# Criar estrutura de diretórios do dbt
for sub_dir in [
    "models/staging",
    "models/intermediate",
    "models/marts"
]:
    os.makedirs(os.path.join(dbt_project_dir, sub_dir), exist_ok=True)

# 1. Configurar dbt_project.yml
dbt_project_content = """name: 'olist_dbt'
version: '1.0.0'
config-version: 2

profile: 'olist_dbt'

model-paths: ["models"]
analysis-paths: ["analyses"]
test-paths: ["tests"]
seed-paths: ["seeds"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]

clean-targets:
  - "target"
  - "dbt_packages"

models:
  olist_dbt:
    staging:
      +materialized: view
    intermediate:
      +materialized: view
    marts:
      +materialized: table
"""

with open(os.path.join(dbt_project_dir, "dbt_project.yml"), "w", encoding="utf-8") as f:
    f.write(dbt_project_content)

# 2. Configurar profiles.yml (em ~/.dbt e localmente em olist_dbt/)
profiles_content = f"""olist_dbt:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: '{db_path}'
      threads: 4
"""

home_dbt_dir = os.path.expanduser("~/.dbt")
os.makedirs(home_dbt_dir, exist_ok=True)

with open(os.path.join(home_dbt_dir, "profiles.yml"), "w", encoding="utf-8") as f:
    f.write(profiles_content)

with open(os.path.join(dbt_project_dir, "profiles.yml"), "w", encoding="utf-8") as f:
    f.write(profiles_content)

print(f"✅ Projeto dbt estruturado em: {dbt_project_dir}")
print(f"✅ Banco DuckDB configurado em: {db_path}")
print(f"✅ profiles.yml gravado em ~/.dbt/profiles.yml e {dbt_project_dir}/profiles.yml")

In [ ]:
# ==============================================================================
# Célula 5: Modelos da Camada Staging (sources.yml, stg_orders.sql, stg_order_items.sql)
# ==============================================================================
import os

base_dir = "/content" if os.path.exists("/content") else os.path.abspath(".")
staging_dir = os.path.join(base_dir, "olist_dbt", "models", "staging")
raw_data_dir_clean = os.path.join(base_dir, "raw_data").replace("\\", "/")

orders_csv = f"{raw_data_dir_clean}/olist_orders_dataset.csv"
items_csv = f"{raw_data_dir_clean}/olist_order_items_dataset.csv"

# 1. Definir sources.yml
sources_content = f"""version: 2

sources:
  - name: raw_olist
    schema: main
    tables:
      - name: olist_orders_dataset
        meta:
          external_location: "read_csv_auto('{orders_csv}')"
      - name: olist_order_items_dataset
        meta:
          external_location: "read_csv_auto('{items_csv}')"
"""

with open(os.path.join(staging_dir, "sources.yml"), "w", encoding="utf-8") as f:
    f.write(sources_content)

# 2. Modelo stg_orders.sql
stg_orders_sql = """with source as (
    select * from {{ source('raw_olist', 'olist_orders_dataset') }}
),

renamed as (
    select
        order_id,
        customer_id,
        order_status,
        cast(order_purchase_timestamp as timestamp) as order_purchase_at,
        cast(order_approved_at as timestamp) as order_approved_at,
        cast(order_delivered_carrier_date as timestamp) as order_delivered_carrier_at,
        cast(order_delivered_customer_date as timestamp) as order_delivered_customer_at,
        cast(order_estimated_delivery_date as timestamp) as order_estimated_delivery_at
    from source
)

select * from renamed
"""

with open(os.path.join(staging_dir, "stg_orders.sql"), "w", encoding="utf-8") as f:
    f.write(stg_orders_sql)

# 3. Modelo stg_order_items.sql
stg_order_items_sql = """with source as (
    select * from {{ source('raw_olist', 'olist_order_items_dataset') }}
),

renamed as (
    select
        order_id,
        cast(order_item_id as integer) as order_item_id,
        product_id,
        seller_id,
        cast(shipping_limit_date as timestamp) as shipping_limit_at,
        cast(price as double) as price,
        cast(freight_value as double) as freight_value
    from source
)

select * from renamed
"""

with open(os.path.join(staging_dir, "stg_order_items.sql"), "w", encoding="utf-8") as f:
    f.write(stg_order_items_sql)

print(f"✅ Camada Staging criada com sucesso em: {staging_dir}")
print("  • sources.yml")
print("  • stg_orders.sql")
print("  • stg_order_items.sql")

In [ ]:
# ==============================================================================
# Célula 6: Modelos das Camadas Intermediate e Marts
# ==============================================================================
import os

base_dir = "/content" if os.path.exists("/content") else os.path.abspath(".")
intermediate_dir = os.path.join(base_dir, "olist_dbt", "models", "intermediate")
marts_dir = os.path.join(base_dir, "olist_dbt", "models", "marts")

# 1. Modelo Intermediate: int_order_items_summary.sql
int_order_items_summary_sql = """with order_items as (
    select * from {{ ref('stg_order_items') }}
),

aggregated as (
    select
        order_id,
        count(order_item_id) as total_items,
        count(distinct product_id) as total_distinct_products,
        round(sum(price), 2) as total_items_amount,
        round(sum(freight_value), 2) as total_freight_amount,
        round(sum(price + freight_value), 2) as total_order_amount
    from order_items
    group by order_id
)

select * from aggregated
"""

with open(os.path.join(intermediate_dir, "int_order_items_summary.sql"), "w", encoding="utf-8") as f:
    f.write(int_order_items_summary_sql)

# 2. Modelo Marts: fct_orders.sql
fct_orders_sql = """with orders as (
    select * from {{ ref('stg_orders') }}
),

items_summary as (
    select * from {{ ref('int_order_items_summary') }}
),

final as (
    select
        o.order_id,
        o.customer_id,
        o.order_status,
        o.order_purchase_at,
        o.order_approved_at,
        o.order_delivered_carrier_at,
        o.order_delivered_customer_at,
        o.order_estimated_delivery_at,
        
        -- Métricas agregadas financeiras e volumétricas
        coalesce(i.total_items, 0) as total_items,
        coalesce(i.total_distinct_products, 0) as total_distinct_products,
        coalesce(i.total_items_amount, 0.0) as total_items_amount,
        coalesce(i.total_freight_amount, 0.0) as total_freight_amount,
        coalesce(i.total_order_amount, 0.0) as total_order_amount,
        
        -- Lead time de entrega em dias corridos
        round(date_diff('hour', o.order_purchase_at, o.order_delivered_customer_at) / 24.0, 2) as delivery_time_days,
        
        -- Margem de cumprimento do prazo previsto (positivo = entrega antes da estimativa; negativo = atraso)
        round(date_diff('hour', o.order_delivered_customer_at, o.order_estimated_delivery_at) / 24.0, 2) as delivery_margin_days

    from orders o
    left join items_summary i on o.order_id = i.order_id
)

select * from final
"""

with open(os.path.join(marts_dir, "fct_orders.sql"), "w", encoding="utf-8") as f:
    f.write(fct_orders_sql)

print("✅ Modelos Intermediate e Marts criados com sucesso:")
print(f"  • {os.path.join(intermediate_dir, 'int_order_items_summary.sql')}")
print(f"  • {os.path.join(marts_dir, 'fct_orders.sql')}")

In [ ]:
# ==============================================================================
# Célula 7: Configuração dos Testes de Qualidade de Dados (dbt Tests)
# ==============================================================================
import os

base_dir = "/content" if os.path.exists("/content") else os.path.abspath(".")
staging_dir = os.path.join(base_dir, "olist_dbt", "models", "staging")
marts_dir = os.path.join(base_dir, "olist_dbt", "models", "marts")

# 1. Testes de Staging: stg_orders.yml
stg_tests_yaml = """version: 2

models:
  - name: stg_orders
    description: "Tabela de staging dos pedidos com colunas limpas e tipadas."
    columns:
      - name: order_id
        description: "Identificador único do pedido."
        tests:
          - unique
          - not_null
      - name: customer_id
        description: "Chave estrangeira do cliente."
        tests:
          - not_null
      - name: order_status
        description: "Status do ciclo de vida do pedido no marketplace."
        tests:
          - not_null
          - accepted_values:
              values:
                - delivered
                - shipped
                - canceled
                - invoiced
                - processing
                - unavailable
                - created
                - approved

  - name: stg_order_items
    description: "Tabela de staging dos itens comprados em cada pedido."
    columns:
      - name: order_id
        description: "Identificador do pedido pai."
        tests:
          - not_null
      - name: order_item_id
        description: "Sequencial do item no pedido."
        tests:
          - not_null
      - name: price
        description: "Preço unitário do produto."
        tests:
          - not_null
"""

with open(os.path.join(staging_dir, "stg_orders.yml"), "w", encoding="utf-8") as f:
    f.write(stg_tests_yaml)

# 2. Testes de Marts: marts.yml
marts_tests_yaml = """version: 2

models:
  - name: fct_orders
    description: "Tabela fato consolidada de pedidos com métricas de valor e entrega."
    columns:
      - name: order_id
        description: "Chave primária única do pedido."
        tests:
          - unique
          - not_null
      - name: order_status
        description: "Status do pedido."
        tests:
          - not_null
      - name: total_order_amount
        description: "Valor total consolidado do pedido (itens + frete)."
        tests:
          - not_null
"""

with open(os.path.join(marts_dir, "marts.yml"), "w", encoding="utf-8") as f:
    f.write(marts_tests_yaml)

print("✅ Testes de Data Quality configurados com sucesso:")
print(f"  • {os.path.join(staging_dir, 'stg_orders.yml')}")
print(f"  • {os.path.join(marts_dir, 'marts.yml')}")

In [ ]:
# ==============================================================================
# Célula 8: Execução do Pipeline (dbt run) e Validação de Qualidade (dbt test)
# ==============================================================================
import os

base_dir = "/content" if os.path.exists("/content") else os.path.abspath(".")
dbt_project_dir = os.path.join(base_dir, "olist_dbt")

print("=" * 80)
print("🚀 ETAPA 1: EXECUTANDO AS TRANSFORMAÇÕES (dbt run)...")
print("=" * 80)
!cd "{dbt_project_dir}" && dbt run --profiles-dir .

print("\n" + "=" * 80)
print("🧪 ETAPA 2: VALIDANDO OS CONTRATOS E QUALIDADE DE DADOS (dbt test)...")
print("=" * 80)
!cd "{dbt_project_dir}" && dbt test --profiles-dir .

In [ ]:
# ==============================================================================
# Célula 9: Consumo Analítico Direto do DuckDB com Python & Pandas
# ==============================================================================
import os
import duckdb
import pandas as pd

base_dir = "/content" if os.path.exists("/content") else os.path.abspath(".")
db_path = os.path.join(base_dir, "olist_analytics.duckdb")

# Conectar ao banco analítico DuckDB em modo somente leitura
con = duckdb.connect(database=db_path, read_only=True)

print("=" * 80)
print("📊 1. TABELAS E VIEWS DISPONÍVEIS NO DUCKDB")
print("=" * 80)
display(con.execute("SHOW TABLES;").df())

print("\n" + "=" * 80)
print("💰 2. TOP 5 PEDIDOS COM MAIOR VALOR TOTAL (fct_orders)")
print("=" * 80)
query_top_orders = """
SELECT 
    order_id,
    order_status,
    total_items,
    total_items_amount,
    total_freight_amount,
    total_order_amount,
    delivery_time_days
FROM fct_orders
WHERE total_order_amount > 0
ORDER BY total_order_amount DESC
LIMIT 5;
"""
display(con.execute(query_top_orders).df())

print("\n" + "=" * 80)
print("📈 3. CONSOLIDAÇÃO EXECUTIVA POR STATUS DE PEDIDO")
print("=" * 80)
query_status = """
SELECT 
    order_status,
    COUNT(order_id) AS total_pedidos,
    ROUND(SUM(total_order_amount), 2) AS receita_total_reais,
    ROUND(AVG(total_order_amount), 2) AS ticket_medio_reais,
    ROUND(AVG(delivery_time_days), 1) AS prazo_medio_entrega_dias
FROM fct_orders
GROUP BY order_status
ORDER BY total_pedidos DESC;
"""
display(con.execute(query_status).df())

print("\n" + "=" * 80)
print("📅 4. SÉRIE TEMPORAL MENSAL DE RECEITA E PEDIDOS ENTREGUES (2017-2018)")
print("=" * 80)
query_monthly = """
SELECT 
    strftime(order_purchase_at, '%Y-%m') AS mes_ano,
    COUNT(order_id) AS total_pedidos,
    ROUND(SUM(total_order_amount), 2) AS receita_total,
    ROUND(AVG(total_order_amount), 2) AS ticket_medio
FROM fct_orders
WHERE order_purchase_at >= '2017-01-01' AND order_purchase_at < '2018-09-01'
  AND order_status = 'delivered'
GROUP BY 1
ORDER BY 1;
"""
df_monthly = con.execute(query_monthly).df()
display(df_monthly)

con.close()
print("\n🔒 Conexão com o DuckDB finalizada com sucesso.")

# 🏁 Conclusão e Melhores Práticas de Analytics Engineering

---

### 💡 Principais Benefícios Demonstrados
1. **Pilha Analítica Serverless e de Custo Zero:**
   - Com o **DuckDB**, eliminamos a necessidade de instanciar data warehouses caros (BigQuery, Snowflake, Redshift) para exploração, desenvolvimento local, prototipação ou validação em pipelines de CI/CD.
2. **Engenharia de Analytics Modular com dbt:**
   - **Camada Staging:** Isolamento de regras de ingestão e tipagem em relação às fontes cruas.
   - **Camada Intermediate:** Eliminação de joins e agregações complexas repetitivas.
   - **Camada Marts:** Entrega de datasets finais enriquecidos, prontos para consumo por áreas de negócio.
3. **Data Quality como Código:**
   - Os testes de schema (`unique`, `not_null`, `accepted_values`) garantem que desvios de dados (*data drift* ou anomalias) sejam identificados antes que afetem dashboards de BI e decisões estratégicas.

### 🚀 Extensões Sugeridas
- Adicionar as dimensões de clientes (`dim_customers`) e produtos (`dim_products`) a partir dos demais arquivos do dataset Olist.
- Criar métricas dbt (`dbt Semantic Layer`) para padronização de KPIs (CAC, LTV, Churn).
- Automatizar a execução periódica com GitHub Actions ou orquestradores leves (Mage, Prefect, Airflow).